# Batch 4 — D01, D02, E24, E30, E29 (validation only; no deep training)

**Runtime:** any runtime (CPU is enough; the L4 runtime is fine and has more CPU cores). Runtime → Run all; about **45–75 min**,
almost all of it E29 (LightGBM and XGBoost on 343 inputs, 5 windows × 3 horizons × 3 seeds). If Colab disconnects, Run all again:
finished E29 windows are skipped.

Rules and expectations fixed before any run: `claude/batches-4-7-design-2026-10-16.md`.

| Item | What |
|---|---|
| D01 | Class shares, training vs validation, per horizon and window |
| D02 | Date-block bootstrap 95% intervals (blocks of 10 trading days) for E21, XGBoost, the floors and the differences |
| E24 | E21 + XGBoost probabilities 50/50 (official + 4 walk-forward windows, from saved predictions) |
| E30 | Two class biases for macro-F1, fitted on the 4 walk-forward windows only; leave-one-window-out check |
| E29 | LightGBM on 29 inputs + 157 Alpha158-style factors + 157 daily cross-sectional ranks; XGBoost on the same 343 as the check |

Outputs in `MyDrive/Senior/results/batch4/`. Test block stays sealed.

In [ ]:
!nvidia-smi -L || echo 'no GPU: fine for batch 4'
!nproc

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, json, os, subprocess, sys
for f in ['scripts/posthoc_eval.py', 'scripts/select_final.py', 'scripts/factors_tabular.py', 'src/alpha158.py', 'src/features_vol.py']:
    assert os.path.exists(f), f'stop: push the batch 4-7 code first ({f} missing)'
for f in ['scripts/baselines_deep.py', 'scripts/posthoc_eval.py', 'src/multiscale_patchtst.py']:
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
FOLDS = [('2022H2', '2022-07-01', '2022-12-31'), ('2023H1', '2023-01-01', '2023-06-30'),
         ('2023H2', '2023-07-01', '2023-12-31'), ('2024H1', '2024-01-01', '2024-06-30')]
B3, B4, B5, B6, B7 = (f'{OUT_DIR}/batch{i}' for i in (3, 4, 5, 6, 7))
E21_JSON = f'{B3}/e21_stacked.json'
need = [E21_JSON, E21_JSON.replace('.json', '_preds.csv.gz'), f'{OUT_DIR}/xgb_e03.json', f'{OUT_DIR}/xgb_e03_preds.csv.gz']
need += [f'{B3}/e22/{f}_deep{s}' for f, _, _ in FOLDS for s in ('.json', '_preds.csv.gz')]
need += [f'{OUT_DIR}/e05/{f}_xgb{s}' for f, _, _ in FOLDS for s in ('.json', '_preds.csv.gz')]
need += []
for f in need:
    assert os.path.exists(f), f'stop: missing {f}'
for d in [B4, f'{B4}/e29']:
    os.makedirs(d, exist_ok=True)
print('earlier files found; batch 4 outputs go to', B4)

In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
try:
    import lightgbm
except ImportError:
    !pip -q install lightgbm
    import lightgbm
print('lightgbm', lightgbm.__version__)

In [ ]:
# Tests; the notebook stops here if any fails (about 8 minutes)
for t in ['tests/test_pipeline.py', 'tests/test_no_lookahead.py', 'tests/test_new_features.py', 'tests/test_posthoc.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-1500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed\n' + r.stderr[-2000:]

In [ ]:
# E29: one process, factors computed once; windows already written are skipped
W = ' '.join(f'--window {f} {s} {e}' for f, s, e in FOLDS)
!python scripts/factors_tabular.py --out-dir "$B4/e29" --skip-done --window official {W}

In [ ]:
# D01, D02, E24, E30 and the E29 verdict -> batch4_summary.json (keep flags read by batch 7)
DW = ' '.join(f'--deep-window {f} "{B3}/e22/{f}_deep_preds.csv.gz"' for f, _, _ in FOLDS)
XW = ' '.join(f'--xgb-window {f} "{OUT_DIR}/e05/{f}_xgb_preds.csv.gz"' for f, _, _ in FOLDS)
WJ = ' '.join(f'--window-json {f} "{B3}/e22/{f}_deep.json"' for f, _, _ in FOLDS)
!python scripts/posthoc_eval.py batch4 --deep-official "$B3/e21_stacked_preds.csv.gz" {DW} --xgb-official "$OUT_DIR/xgb_e03_preds.csv.gz" {XW} \
    --e29-dir "$B4/e29" --official-json "$E21_JSON" {WJ} --out "$B4/batch4_summary.json"
print(json.dumps(json.load(open(f'{B4}/batch4_summary.json'))['keep'], indent=1))